In [ ]:
# !pip uninstall torch torchvision torchaudio whisperx -y

# !pip install torch torchaudio --index-url https://download.pytorch.org/whl/cpu

# !pip install git+https://github.com/m-bain/whisperx.git

# !pip install pyannote.audio

# !pip install --upgrade whisperx torch

In [ ]:
import whisperx
import torch

# === 설정 ===
# AUDIO_FILE = "./test_90sec.mp3"  # 분석할 오디오 파일 경로
AUDIO_FILE = "./tuition_fee_speech.mp3"  # 분석할 오디오 파일 경로
DEVICE = "cpu"  # M1/M2/M3 Mac은 "mps" 미지원
LANGUAGE = "ko"  # 한국어

# === WhisperX 모델 로딩 ===
print("WhisperX 모델 로딩 중...")
model = whisperx.load_model(
    "large",  # tiny/small/medium/large 중 선택
    device=DEVICE,
    language=LANGUAGE,
    compute_type="int8",  # int8 / float16 / float32 중 선택
    vad_method="silero"   # 🔸 pyannote 대신 silero 사용
)

# === 음성 인식 ===
print("음성 인식 중...")
asr_result = model.transcribe(AUDIO_FILE)

# === 결과 출력 ===
print("\n 음성 인식 결과:\n")
for seg in asr_result["segments"]:
    start = seg["start"]
    end = seg["end"]
    text = seg["text"].strip()
    print(f"[{start:.2f} - {end:.2f}] {text}")

🌀 WhisperX 모델 로딩 중...


c:\Users\Playdata\anaconda3\envs\ollama_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


>>Performing voice activity detection using Silero...


Using cache found in C:\Users\Playdata/.cache\torch\hub\snakers4_silero-vad_master


📝 음성 인식 중...


# 요약 모델 

In [ ]:
import whisperx
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# === 설정 ===
AUDIO_FILE = "test_90sec.mp3"  # 분석할 오디오 파일 경로
DEVICE = "cpu"  # M1/M2/M3 Mac은 "mps" 미지원
LANGUAGE = "ko"  # 한국어

# === WhisperX 모델 로딩 ===
print("WhisperX 모델 로딩 중...")
model = whisperx.load_model(
    "medium",  # tiny/small/medium/large 중 선택
    device=DEVICE,
    language=LANGUAGE,
    compute_type="int8",  # int8 / float16 / float32 중 선택
    vad_method="silero"   # 🔸 pyannote 대신 silero 사용
)

# === 음성 인식 ===
print("📝 음성 인식 중...")
asr_result = model.transcribe(AUDIO_FILE)

# === 음성 인식 결과 출력 ===
print("\n📢 음성 인식 결과:\n")
transcribed_text = " ".join([seg["text"].strip() for seg in asr_result["segments"]])

# 요약할 텍스트 (음성 인식된 텍스트)
print(f"\n음성 인식된 텍스트: \n{transcribed_text}")

# === Qwen 모델 로드 ===
model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,  # FP16을 사용하여 메모리 사용 최적화
    device_map="auto",  # 자동으로 디바이스 분산
    trust_remote_code=True
)

print("Qwen 모델 로드 완료!")

# === 텍스트 요약 ===
inputs = tokenizer(f"요약해 주세요: {transcribed_text}", return_tensors="pt").to(device="cuda" if torch.cuda.is_available() else "cpu")
outputs = model.generate(inputs["input_ids"], max_length=150, num_beams=4, no_repeat_ngram_size=2, early_stopping=True)

# 요약된 텍스트 출력
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n 요약 결과:\n")
print(summary)

WhisperX 모델 로딩 중...
>>Performing voice activity detection using Silero...


Using cache found in C:\Users\Playdata/.cache\torch\hub\snakers4_silero-vad_master


📝 음성 인식 중...

📢 음성 인식 결과:


음성 인식된 텍스트: 
이거 행정실에 전화해서 로그인 방법 물어봐야 되는데 지금 전화해보니까 점심시간이라가지고 안 받아 어 나는 그냥 가입 되던데 저도요 저도 그냥 가입 네 그냥 하면 안 돼? 네 그냥 가입 학번 치니까 바로 가입 되더라고요. 아이디랑 비밀번호 입력하고.


c:\Users\Playdata\anaconda3\envs\vectordb_env\lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--Qwen--Qwen2.5-7B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be 

Qwen 모델 로드 완료!


In [ ]:

def make_prompt(transcribed_text):
    prompt = f"""
당신은 회의록 작성 전문가입니다. 아래는 한국어로 된 회의 음성 인식 결과입니다. 이를 바탕으로 회의록을 다음 포맷에 맞게 작성해 주세요.
다음은 음성에서 텍스트로 변환된 원문입니다. 이 텍스트는 말의 흐름에 따라 작성되어 있으며, 반복, 군더더기, 비문 등이 포함되어 있을 수 있습니다.


## 지침:

- 핵심 내용을 파악하고 요점을 간결하게 정리하십시오.
- 말투나 반복, 불필요한 감탄사 등은 제거하십시오.
- 정제된 서면 문장 형태로 작성하십시오.
- 원문의 흐름을 보존하되, 문맥이 자연스럽고 명확하도록 재구성하십시오.
- 필요한 경우, 항목별로 정리하거나 문단을 나눠 구조화하십시오.
- 문체는 뉴스 기사나 공식 보고서처럼 객관적이고 깔끔하게 유지하십시오.
- 요약 분량은 원문 길이에 비례하여 적절하게 조절하십시오.
- 요약문은 한자, 영어 쓰지 말고 한글로만 작성하세요.
- 원문에 포함된 인물, 장소, 사건 등은 요약문에서도 언급하십시오.

## 예시:
[회의 개요]
- 일자:
- 주최자:
- 참석자:
- 회의 주제:

[주요 논의 사항]
- 

[핵심 키워드]
- 

[주요 결정 사항]
- 

[액션 아이템]
- 

[향후 일정]
- 

[회의 내용 요약]


{transcribed_text}
"""
    return prompt


In [8]:
!pip install whisperx


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ------------------------------- -------- 13.1/16.5 MB 68.7 MB/s eta 0:00:01
   ---------------------------------------- 16.5/16.5 MB 54.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/19.3 MB ? eta -:--:--
   -- ------------------------------------- 1.3/19.3 MB 6.1 MB/s eta 0:00:03
   ---- ----------------------------------- 2.1/19.3 MB 5.3 MB/s eta 0:00:04
   ----- ---------------------------------- 2.9/19.3 MB 4.1 MB/s eta 0:00:05
   -------------------------- ------------- 12.8/19.3 MB 15.2 MB/s eta 0:00:01
   ------------------------------------ --- 17.8/19.3 MB 16.8 MB/s eta 0:00:01
   ----

# ollama -> qwen:7b

In [16]:
import whisperx
import ollama

# WhisperX 로딩
audio_file = "./test_90sec.mp3"
device = "cpu"
language = "ko"

print("WhisperX 모델 로딩 중...")
model = whisperx.load_model(
    "medium",
    device=device,
    language=language,
    compute_type="int8",
    vad_method="silero"
)

# 음성 인식
print("📝 음성 인식 중...")
asr_result = model.transcribe(audio_file)

# 텍스트 추출
result = " ".join([seg["text"].strip() for seg in asr_result["segments"]])

# 프롬프트 구성 함수 (예시)
def make_prompt(text):
    return f"다음 내용을 요약해 주세요:\n{text}"

# Ollama로 Qwen2.5 모델에 요약 요청
response = ollama.generate(model='qwen:7b', prompt=make_prompt(result))

# 출력
print("\n📄 요약 결과:")
print(response['response'])  # Ollama의 응답 본문


WhisperX 모델 로딩 중...
>>Performing voice activity detection using Silero...


Using cache found in C:\Users\Playdata/.cache\torch\hub\snakers4_silero-vad_master


📝 음성 인식 중...

📄 요약 결과:
이 내용은某人在行政室通过电话询问登录方法的过程。最初，打电话的人想要确认之前简单的注册流程是否依然有效。通话中，对方提到只需按照之前的步骤操作：输入学号、身份证信息（包含ID和密码），然后完成身份验证。

总结来说，这个情景是关于如何再次登录某个系统，而具体的操作步骤与之前的注册流程相似。


---------

# ollama -> qwen2.5


In [18]:
import whisperx
import ollama

# WhisperX 설정
audio_file = "./test_90sec.mp3"
device = "cpu"
language = "ko"

print("WhisperX 모델 로딩 중...")
model = whisperx.load_model(
    "medium",
    device=device,
    language=language,
    compute_type="int8",
    vad_method="silero"
)

print("📝 음성 인식 중...")
asr_result = model.transcribe(audio_file)

# 텍스트 추출
result = " ".join([seg["text"].strip() for seg in asr_result["segments"]])

# 프롬프트 구성 함수 (구어체 정제형 프롬프트 적용)
def make_prompt(transcribed_text):
    prompt = f"""
다음은 음성에서 텍스트로 변환된 원문입니다. 이 텍스트는 말의 흐름에 따라 작성되어 있으며, 반복, 군더더기, 비문 등이 포함되어 있을 수 있습니다.

당신의 임무는 이 텍스트를 이해하기 쉬운 요약문으로 정리하는 것입니다.

## 지침:

- 핵심 내용을 파악하고 요점을 간결하게 정리하십시오.
- 말투나 반복, 불필요한 감탄사 등은 제거하십시오.
- 정제된 서면 문장 형태로 작성하십시오.
- 원문의 흐름을 보존하되, 문맥이 자연스럽고 명확하도록 재구성하십시오.
- 필요한 경우, 항목별로 정리하거나 문단을 나눠 구조화하십시오.
- 문체는 뉴스 기사나 공식 보고서처럼 객관적이고 깔끔하게 유지하십시오.
- 요약문은 한자, 영어 쓰지 말고 한글로만 작성하세요.
- 원문에 포함된 인물, 장소, 사건 등은 요약문에서도 언급하십시오.

## 예시:

**원문**:  
"네, 그래서 제가 그걸 이제 계속 고민을 하다가 결국에는 이 방향으로 결정을 했거든요. 아, 그리고 뭐 다른 분들도 의견을 주셨고요. 네. 그래서 그렇게 진행하기로 했습니다."

**요약**:  
해당 화자는 여러 사람의 의견을 수렴한 끝에 최종적으로 특정 방향으로 결정했음을 밝히고 있다.

---

이제 아래 텍스트를 요약하십시오:

{transcribed_text}
"""
    return prompt

# Ollama 요약 요청
response = ollama.generate(model='qwen2.5', prompt=make_prompt(result))

# 출력
print("\n📄 요약 결과:")
print(response['response'])


WhisperX 모델 로딩 중...
>>Performing voice activity detection using Silero...


Using cache found in C:\Users\Playdata/.cache\torch\hub\snakers4_silero-vad_master


📝 음성 인식 중...

📄 요약 결과:
행정실에 전화해 로그인 방법을 물었으나 점심시간이라 받아주지 않았습니다. 그래서 직접 가입했고, 학번만 입력한 뒤 바로 아이디와 비밀번호가 생성되었습니다.
